# 0. Data source
Set `USE_RAW` below. With `USE_RAW = False` (default), all Part 1 cells are skipped and the analysis uses the shipped consolidated `*_full.csv`.

In [ ]:
# Data source switch
#   USE_RAW = True : run Part 1 -- merge the raw per-run simulation outputs under
#                    2_prelimit_simulation/6_ECC_saved_results_full/ (full local results, NOT shipped)
#                    and write the consolidated *_full.csv to cpp_results_formal_run/...;
#                    Part 2 then reads them from there.
#   USE_RAW = False: skip Part 1; Part 2 reads the shipped consolidated *_full.csv
#                    from 2_prelimit_simulation/4_ECC_saved_results/.
USE_RAW = False

# 1. Merge Data

## Paths

In [ ]:
import pandas as pd
from pathlib import Path
import re
import os
import numpy as np



# search only under this directory
root = Path(
    "../../2_prelimit_simulation/6_ECC_saved_results_full/2_ECC10_benchmarks_5000p"
)
model_name = "CC10"
eps_band = "Out"
results_folder_name = ("cpp_results_formal_run/2_ECC10_benchmarks_5000p" if USE_RAW
                       else "../../2_prelimit_simulation/4_ECC_saved_results/2_ECC10_benchmarks_5000p")
figures_folder_name = "cpp_results_formal_run/2_ECC10_benchmarks_5000p"

if not os.path.exists(results_folder_name):
    os.makedirs(results_folder_name)
if not os.path.exists(figures_folder_name):
    os.makedirs(figures_folder_name)

dim_K = 10

In [ ]:
if USE_RAW:
    from pathlib import Path
    import pandas as pd

    ROOT = Path("../../2_prelimit_simulation/6_ECC_saved_results_full/2_ECC10_benchmarks_5000p/prelimit_simulation_results/benchmark_results")
    rows = []
    for bmk_dir in sorted(ROOT.iterdir()):
        if not bmk_dir.is_dir(): continue
        for combo_dir in sorted(bmk_dir.iterdir()):
            if not combo_dir.is_dir(): continue
            iters = [p for p in combo_dir.iterdir()
                     if p.is_dir() and p.name.startswith("iteration")]
            has_last = any(p.name == "iteration_last" for p in iters)
            rows.append({
                "benchmark": bmk_dir.name,
                "combo": combo_dir.name,
                "n_iters": len(iters),
                "terminated_normally": has_last,
            })
    df = pd.DataFrame(rows)
    target_bmks = ["benchmarkC1", "benchmarkC2", "benchmarkD1", "benchmarkD2"]
    df = df[df['benchmark'].isin(target_bmks)]


    display(df.sort_values("n_iters", ascending=False))
    print(df.groupby("benchmark")["n_iters"].agg(["mean", "median", "max", "count"]))

    # how many times each n_iters appears (grouped by benchmark)
    print("\n=== n_iters distribution per benchmark ===")
    print(df.groupby(["benchmark", "n_iters"]).size().unstack(fill_value=0))

    print("\n=== rows with n_iters = 5 ===")
    display(df[df["n_iters"] == 5])

    print("\n=== rows with terminated_normally = False ===")
    display(df[df["terminated_normally"] == False])

In [ ]:
if USE_RAW:
    from pathlib import Path
    import re
    import numpy as np
    import pandas as pd

    ROOT = Path("../../2_prelimit_simulation/6_ECC_saved_results_full/2_ECC10_benchmarks_5000p/prelimit_simulation_results/benchmark_results")
    # dim_K is already defined as 10 above; reuse it directly here
    N_DELAY = dim_K + 1  # CC10 → 11

    rows_n5 = df[df["n_iters"] == 5].copy()

    # dynamically generate column names by dim_K, fully consistent with your reading script
    def col_names_for(bmk):
        if bmk == "benchmarkC1":
            return (
                [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                + ["lambda","beta_tilde_1","beta_tilde_2","rho"]
                + ["cost","std"]
                + [f"delay{i}" for i in range(1, dim_K + 2)]
                + [f"delay{i}_std" for i in range(1, dim_K + 2)]
            )
        if bmk == "benchmarkC2":
            return (
                [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                + ["lambda","beta_tilde_1","beta_tilde_2","rho"]
                + ["cost","std"]
                + [f"delay{i}" for i in range(1, dim_K + 2)]
                + [f"delay{i}_std" for i in range(1, dim_K + 2)]
            )
        if bmk == "benchmarkD1":
            return (
                [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                + ["lambda","beta_tilde_1","beta_tilde_2"]
                + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                + ["cost","std"]
                + [f"delay{i}" for i in range(1, dim_K + 2)]
                + [f"delay{i}_std" for i in range(1, dim_K + 2)]
            )
        if bmk == "benchmarkD2":
            return (
                [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                + ["lambda","beta_tilde_1","beta_tilde_2"]
                + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                + ["cost","std"]
                + [f"delay{i}" for i in range(1, dim_K + 2)]
                + [f"delay{i}_std" for i in range(1, dim_K + 2)]
            )
        return None

    CSV_NAME = f"CC{dim_K}_final_results.csv"  # CC10_final_results.csv

    def load_one(csv_path, bmk):
        if not csv_path.exists():
            return None
        try:
            return pd.read_csv(csv_path, names=col_names_for(bmk), header=0)
        except pd.errors.EmptyDataError:
            return None

    def best_feasible(df_one):
        """row with minimum cost among those whose delay1..delay_{dim_K+1} are all <= 1; returns None if no feasible solution."""
        if df_one is None or df_one.empty:
            return None
        delay_cols = [f"delay{i}" for i in range(1, N_DELAY + 1)]
        feas = df_one[df_one[delay_cols].le(1).all(axis=1)]
        if feas.empty:
            return None
        return feas.loc[feas["cost"].idxmin()]

    def iter_sort_key(name):
        if name == "iteration_last":
            return (1, 0)
        m = re.match(r"iteration(\d+)$", name)
        return (0, int(m.group(1))) if m else (2, name)

    compare_rows = []
    for _, r in rows_n5.iterrows():
        bmk, combo = r["benchmark"], r["combo"]
        if col_names_for(bmk) is None:
            print(f"[skip] no column names configured for {bmk}, skipping {combo}")
            continue

        base = ROOT / bmk / combo
        iter_dirs = sorted(
            [p for p in base.iterdir() if p.is_dir() and p.name.startswith("iteration")],
            key=lambda p: iter_sort_key(p.name),
        )

        row = {"benchmark": bmk, "combo": combo}
        for it_dir in iter_dirs:
            csv_path = it_dir / "scheduling" / CSV_NAME
            df_it = load_one(csv_path, bmk)
            best  = best_feasible(df_it)
            row[f"cost_{it_dir.name}"] = float(best["cost"]) if best is not None else np.nan
            # too many delay columns, save them all here for full-view inspection
            for i in range(1, N_DELAY + 1):
                row[f"delay{i}_{it_dir.name}"] = float(best[f"delay{i}"]) if best is not None else np.nan

        c_it4  = row.get("cost_iteration4",   np.nan)
        c_last = row.get("cost_iteration_last", np.nan)
        row["delta_last_minus_it4"] = c_last - c_it4
        row["rel_change_%"] = (c_last - c_it4) / c_it4 * 100 if c_it4 else np.nan

        compare_rows.append(row)

    cmp_df = pd.DataFrame(compare_rows)

    iter_order = ["iteration0","iteration1","iteration2","iteration3","iteration4","iteration_last"]
    cost_cols  = [f"cost_{n}"   for n in iter_order if f"cost_{n}"   in cmp_df.columns]
    delay_cols = [f"delay{i}_{n}" for n in iter_order for i in range(1, N_DELAY + 1)
                  if f"delay{i}_{n}" in cmp_df.columns]
    ordered = ["benchmark","combo"] + cost_cols + ["delta_last_minus_it4","rel_change_%"] + delay_cols
    cmp_df = cmp_df[[c for c in ordered if c in cmp_df.columns]]

    # concise view: only look at cost
    view_cost = cmp_df[["benchmark","combo"] + cost_cols + ["delta_last_minus_it4","rel_change_%"]]
    fmt = {c: "{:.4f}" for c in cost_cols}
    fmt["delta_last_minus_it4"] = "{:+.4f}"
    fmt["rel_change_%"] = "{:+.2f}%"
    display(view_cost.style.format(fmt))

    # full view (with all delays)
    print("\n=== full view with delay ===")
    display(cmp_df)

## 1.1. A1-A4:

In [ ]:
if USE_RAW:
    for bmk in ["A1", "A2", "A3", "A4"]:

        pattern = re.compile(
            fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
            fr"rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)_epsilon(?P<epsilon>[^_]+)/pricing/shared_pooled/CC{dim_K}_final_results\.csv"
        )
    
        all_dfs = []
    
        for file_path in root.rglob(f"CC{dim_K}_final_results.csv"):
    
            path_str = str(file_path)
    
            match = pattern.search(path_str)
    
            # keep only those matching your directory structure
            if match:
                # df = pd.read_csv(file_path, header=None)
                if bmk == "A1" or bmk == "A2":
                    col_names = (
                        [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                        + ["rho"] 
                        + ["cost", "std"]
                        + [f"delay{i}" for i in range(1, dim_K + 2)]
                        + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                    )
                elif bmk == "A3" or bmk == "A4":
                    col_names = (
                        [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                        + ["rho"] 
                        + ["cost", "std"]
                        + [f"delay{i}" for i in range(1, dim_K + 2)]
                        + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                    )
                df = pd.read_csv(file_path, names=col_names, header=0)
    
                # print(df.shape)
                # print(df.columns)
                # print(df.head())
    
                epsilon_str = match.group("epsilon")
                df["epsilon"] = float(epsilon_str)
                # df["rho"] = float(match.group("rho"))
                # df["lambda"] = float(match.group("lambda"))
                df["safety"] = float(match.group("safety").replace("n", "-"))
    
                all_dfs.append(df)
    
        # merge
        big_df = pd.concat(all_dfs, ignore_index=True)
    
        print("Number of files read:", len(all_dfs))
        print("Total rows:", len(big_df))
    
        display(big_df.head())
    
        # save
        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_original.csv", index=False)

        if model_name == "RS":

            lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
            n = 400  # scale number between original system and heavy-traffic limit
            delay_bound1, delay_bound3 = 400.0/3, 400.0/3
            b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
            b_vec = np.array([b1, b3])
            print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")

        elif model_name == "CC3":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 90
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "A3" or bmk == "A4" or bmk == "B3" or bmk == "B4" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)

        elif model_name == "CC10":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 240
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "A3" or bmk == "A4":
                for i in range(1, dim_K + 2):  # 1, ..., dim_K+1
                    param_col = f"param_{3 * (i - 1) + 1}"
                    HL_col = f"HL_rate_{i}"
                    denom = b_h if i == dim_K + 1 else b_v            
                    big_df[HL_col] = (big_df[param_col] / big_df["rho"] / denom).round(2)

        print("epsilon:", sorted(big_df["epsilon"].unique()))
        print("rho:", sorted(big_df["rho"].unique()))
        print("safety:", sorted(big_df["safety"].unique()))

        if bmk == "A1" or bmk == "A2":
            param_cols = ["safety", "rho", "epsilon"] + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        elif bmk == "A3" or bmk == "A4":
            param_cols = ["safety", "rho", "epsilon"] + [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
        
        big_df = big_df.sort_values(
            by=param_cols
        ).reset_index(drop=True)

        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv", index=False)
    

## 1.2. B1-B4

In [ ]:
if USE_RAW:
    for bmk in ["B1", "B2", "B3", "B4"]:

        pattern = re.compile(
            fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
            fr"rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)/pricing/shared_pooled/CC{dim_K}_final_results\.csv"
        )
    
        all_dfs = []
    
        for file_path in root.rglob(f"CC{dim_K}_final_results.csv"):
    
            path_str = str(file_path)
    
            match = pattern.search(path_str)
    
            # keep only those matching your directory structure
            if match:
                # df = pd.read_csv(file_path, header=None)
                if bmk == "B1" or bmk == "B2":
                    col_names = (
                        [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                        + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                        + ["cost", "std"]
                        + [f"delay{i}" for i in range(1, dim_K + 2)]
                        + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                    )
                elif bmk == "B3" or bmk == "B4":
                    col_names = (
                        [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                        + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                        + ["cost", "std"]
                        + [f"delay{i}" for i in range(1, dim_K + 2)]
                        + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                    )
                df = pd.read_csv(file_path, names=col_names, header=0)
    
                # print(df.shape)
                # print(df.columns)
                # print(df.head())
    
                # epsilon_str = match.group("epsilon")
                # df["epsilon"] = float(epsilon_str)
                # df["rho"] = float(match.group("rho"))
                # df["lambda"] = float(match.group("lambda"))
                df["safety"] = float(match.group("safety").replace("n", "-"))
    
                all_dfs.append(df)
    
        # merge
        big_df = pd.concat(all_dfs, ignore_index=True)
    
        print("Number of files read:", len(all_dfs))
        print("Total rows:", len(big_df))
    
        display(big_df.head())
    
        # save
        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_original.csv", index=False)

        if model_name == "RS":

            lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
            n = 400  # scale number between original system and heavy-traffic limit
            delay_bound1, delay_bound3 = 400.0/3, 400.0/3
            b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
            b_vec = np.array([b1, b3])
            print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")

        elif model_name == "CC3":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 90
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "B3" or bmk == "B4":
                big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)
            
        elif model_name == "CC10":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 240
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "B3" or bmk == "B4":
                for i in range(1, dim_K + 2):  # 1, ..., dim_K+1
                    param_col = f"param_{3 * (i - 1) + 1}"
                    HL_col = f"HL_rate_{i}"
                    denom = b_h if i == dim_K + 1 else b_v            
                    big_df[HL_col] = (big_df[param_col] / big_df["rho"] / denom).round(2)

        # print("epsilon:", sorted(big_df["epsilon"].unique()))
        print("rho:", sorted(big_df["rho"].unique()))
        print("safety:", sorted(big_df["safety"].unique()))

        if bmk == "B1" or bmk == "B2":
            param_cols = ["safety", "rho"] + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        elif bmk == "B3" or bmk == "B4":
            param_cols = ["safety", "rho"] + [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
        big_df = big_df.sort_values(
            by=param_cols
        ).reset_index(drop=True)

        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv", index=False)
    

## 1.3. C1-C2

In [ ]:
if USE_RAW:
    for bmk in ["C1", "C2"]:
        for scheduling_suffix in ["abs", "rel"]: 

            pattern = re.compile(
                fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
                fr"{scheduling_suffix}_rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)_epsilon(?P<epsilon>[^_]+)/iteration_last/scheduling/CC{dim_K}_final_results\.csv"
            )
        
            all_dfs = []
        
            for file_path in root.rglob(f"CC{dim_K}_final_results.csv"):
        
                path_str = str(file_path)
        
                match = pattern.search(path_str)
        
                # keep only those matching your directory structure
                if match:
                    # df = pd.read_csv(file_path, header=None)
                    if bmk == "C1":
                        col_names = (
                            [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                            + ["lambda","beta_tilde_1","beta_tilde_2"]
                            + ["rho"] 
                            + ["cost", "std"]
                            + [f"delay{i}" for i in range(1, dim_K + 2)]
                            + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                        )
                    elif bmk == "C2":
                        col_names = (
                            [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                            + ["lambda","beta_tilde_1","beta_tilde_2"]
                            + ["rho"] 
                            + ["cost", "std"]
                            + [f"delay{i}" for i in range(1, dim_K + 2)]
                            + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                        )
                    df = pd.read_csv(file_path, names=col_names, header=0)
        
                    # print(df.shape)
                    # print(df.columns)
                    # print(df.head())
        
                    epsilon_str = match.group("epsilon")
                    df["epsilon"] = float(epsilon_str)
                    # df["rho"] = float(match.group("rho"))
                    # df["lambda"] = float(match.group("lambda"))
                    df["safety"] = float(match.group("safety").replace("n", "-"))
        
                    all_dfs.append(df)
        
            # merge
            big_df = pd.concat(all_dfs, ignore_index=True)
        
            print("Number of files read:", len(all_dfs))
            print("Total rows:", len(big_df))
        
            display(big_df.head())
        
            # save
            big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_original.csv", index=False)
    
            if model_name == "RS":
    
                lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
                n = 400  # scale number between original system and heavy-traffic limit
                delay_bound1, delay_bound3 = 400.0/3, 400.0/3
                b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
                b_vec = np.array([b1, b3])
                print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")
    
                if bmk == "C1_abs" or bmk == "C1_rel" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D1_abs" or bmk == "D1_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b1).round(2)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b3).round(2)
    
            elif model_name == "CC3":
                lambda_ast = 1.0/2*0.95
                n = 400  # scale number between original system and heavy-traffic limit
                inv_sqrt_n = 1 / np.sqrt(n)
                delay_bound_v, delay_bound_h = 60, 90
                b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
    
                if bmk == "C1_abs" or bmk == "C1_rel" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D1_abs" or bmk == "D1_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b_v).round(3)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b_h).round(3)
                
                if bmk == "A3" or bmk == "A4" or bmk == "B3" or bmk == "B4" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)
    
            elif model_name == "CC10":
                lambda_ast = 1.0/2*0.95
                n = 400  # scale number between original system and heavy-traffic limit
                inv_sqrt_n = 1 / np.sqrt(n)
                delay_bound_v, delay_bound_h = 60, 240
                b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
                if bmk == "C1" or bmk == "C2" or bmk == "D1" or bmk == "D2":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b_v).round(2)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b_h).round(2)
                if bmk == "C2":
                    for i in range(1, dim_K + 2):  # 1, ..., dim_K+1
                        param_col = f"param_{3 * (i - 1) + 1}"
                        HL_col = f"HL_rate_{i}"
                        denom = b_h if i == dim_K + 1 else b_v            
                        big_df[HL_col] = (big_df[param_col] / big_df["rho"] / denom).round(2)
    
            print("epsilon:", sorted(big_df["epsilon"].unique()))
            print("rho:", sorted(big_df["rho"].unique()))
            print("safety:", sorted(big_df["safety"].unique()))
    
            if bmk == "C1":
                param_cols = ["safety", "rho", "epsilon", "lambda", "beta_rate_1", "beta_rate_2"] + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            elif bmk == "C2":
                param_cols = ["safety", "rho", "epsilon", "lambda", "beta_rate_1", "beta_rate_2"] + [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
            
            big_df = big_df.sort_values(
                by=param_cols
            ).reset_index(drop=True)
    
            big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv", index=False)
    

## 1.4. D1-D2:

In [ ]:
if USE_RAW:
    for bmk in ["D1", "D2"]:
        for scheduling_suffix in ["abs", "rel"]: 

            pattern = re.compile(
                fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
                fr"{scheduling_suffix}_rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)/iteration_last/scheduling/CC{dim_K}_final_results\.csv"
            )
        
            all_dfs = []
        
            for file_path in root.rglob(f"CC{dim_K}_final_results.csv"):
        
                path_str = str(file_path)
        
                match = pattern.search(path_str)
        
                # keep only those matching your directory structure
                if match:
                    # df = pd.read_csv(file_path, header=None)
                    if bmk == "D1":
                        col_names = (
                            [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                            + ["lambda","beta_tilde_1","beta_tilde_2"]
                            + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                            + ["cost", "std"]
                            + [f"delay{i}" for i in range(1, dim_K + 2)]
                            + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                        )
                    elif bmk == "D2":
                        col_names = (
                            [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
                            + ["lambda","beta_tilde_1","beta_tilde_2"]
                            + ["rho"] + [f"rho_{i}" for i in range(2, dim_K + 2)]
                            + ["cost", "std"]
                            + [f"delay{i}" for i in range(1, dim_K + 2)]
                            + [f"delay{i}_std" for i in range(1, dim_K + 2)]
                        )
                    df = pd.read_csv(file_path, names=col_names, header=0)
        
                    # print(df.shape)
                    # print(df.columns)
                    # print(df.head())
        
                    # epsilon_str = match.group("epsilon")
                    # df["epsilon"] = float(epsilon_str)
                    # df["rho"] = float(match.group("rho"))
                    # df["lambda"] = float(match.group("lambda"))
                    df["safety"] = float(match.group("safety").replace("n", "-"))
        
                    all_dfs.append(df)
        
            # merge
            big_df = pd.concat(all_dfs, ignore_index=True)
        
            print("Number of files read:", len(all_dfs))
            print("Total rows:", len(big_df))
        
            display(big_df.head())
        
            # save
            big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_original.csv", index=False)
    
            if model_name == "RS":
    
                lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
                n = 400  # scale number between original system and heavy-traffic limit
                delay_bound1, delay_bound3 = 400.0/3, 400.0/3
                b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
                b_vec = np.array([b1, b3])
                print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")
    
                if bmk == "C1_abs" or bmk == "C1_rel" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D1_abs" or bmk == "D1_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b1).round(2)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b3).round(2)
    
            elif model_name == "CC3":
                lambda_ast = 1.0/2*0.95
                n = 400  # scale number between original system and heavy-traffic limit
                inv_sqrt_n = 1 / np.sqrt(n)
                delay_bound_v, delay_bound_h = 60, 90
                b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
    
                if bmk == "C1_abs" or bmk == "C1_rel" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D1_abs" or bmk == "D1_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b_v).round(3)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b_h).round(3)
    
                if bmk == "A3" or bmk == "A4" or bmk == "B3" or bmk == "B4" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                    big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                    big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)
    
            elif model_name == "CC10":
                lambda_ast = 1.0/2*0.95
                n = 400  # scale number between original system and heavy-traffic limit
                inv_sqrt_n = 1 / np.sqrt(n)
                delay_bound_v, delay_bound_h = 60, 240
                b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
                if bmk == "C1" or bmk == "C2" or bmk == "D1" or bmk == "D2":
                    big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b_v).round(3)
                    big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b_h).round(3)
                if bmk == "D2":
                    for i in range(1, dim_K + 2):  # 1, ..., dim_K+1
                        param_col = f"param_{3 * (i - 1) + 1}"
                        HL_col = f"HL_rate_{i}"
                        denom = b_h if i == dim_K + 1 else b_v            
                        big_df[HL_col] = (big_df[param_col] / big_df["rho"] / denom).round(2)
    
            # print("epsilon:", sorted(big_df["epsilon"].unique()))
            print("rho:", sorted(big_df["rho"].unique()))
            print("safety:", sorted(big_df["safety"].unique()))
    
            if bmk == "D1":
                param_cols = ["safety", "rho", "lambda", "beta_rate_1", "beta_rate_2"] + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            elif bmk == "D2":
                param_cols = ["safety", "rho", "lambda", "beta_rate_1", "beta_rate_2"] + [f"param_{i}" for i in range(1, 3 * (dim_K + 1) + 1)]
            
            big_df = big_df.sort_values(
                by=param_cols
            ).reset_index(drop=True)
    
            big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv", index=False)

# 2. Check Optimal Row

## Function: 

In [ ]:
def optimal_check(df):
    # 1. optimal parameter row

    print("Total parameter combinations:")
    print(df.shape)
    
    df_valid = df[
        df[[f"delay{i}" for i in range(1, dim_K + 2)]].le(1).all(axis=1)
    ]
    if df_valid.empty:
        print("df_valid is empty!")
        return df_valid, []
    else:
    
        print("Feasible solutions (delay < 1%):")
        print(df_valid.shape)
        print("\n")
        
        best_row = df_valid.loc[df_valid['cost'].idxmin()]

        best_row_copy = best_row.copy()
        for i in range(1, dim_K + 2):
            best_row_copy[[f"delay{i}", f"delay{i}_std"]] = best_row_copy[[f"delay{i}", f"delay{i}_std"]].round(2)
        
        best_row_copy[["cost", "std"]] = best_row_copy[["cost", "std"]].round(2)
        print(",".join(best_row_copy[show_names].astype(str).values))
        
        print("Optimal row:")
        display(best_row[show_names].to_frame().T)
        print("------------------------------------------------------------------------------")
        print(best_row[show_names])
        print("------------------------------------------------------------------------------")

    return best_row, np.sort(df["rho"].unique())

In [ ]:
import pandas as pd

df_benchmarks = pd.DataFrame(columns=["rho", "rho_min", "rho_max", "rho_grid", "cost", "std"] + [f"delay{i}" for i in range(1, dim_K + 2)])

# df_benchmarks_s1 = pd.DataFrame(columns=["rho", "rho_min", "rho_max", "rho_grid", "cost", "std"] + [f"delay{i}" for i in range(1, dim_K + 2)])

## 2.1. A1-A4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ['A1', 'A2', 'A3', 'A4']:

    if bmk == "A1" or bmk == "A2":
        show_names = (["safety", "rho", "epsilon"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                      + ["cost", "std"]
                      + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (["safety", "rho", "epsilon"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        )
    elif bmk == "A3" or bmk == "A4":
        show_names = (
            ["safety", "rho", "epsilon"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
            + ["cost", "std"]
            + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (
            ["safety", "rho", "epsilon"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
        )    
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    best_row, rho_list = optimal_check(df)

    df_benchmarks.loc[bmk] = (
        [
            best_row["rho"],
            rho_list[0],
            rho_list[-1],
            (rho_list[1:] - rho_list[:-1]).min().round(2),
            best_row["cost"],
            best_row["std"],
        ]
        + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
    )

    # best_row, rho_list = optimal_check(df[df['safety'] == 1.0])

    # if best_row.empty == False:
    #     df_benchmarks_s1.loc[bmk] = (
    #         [
    #             best_row["rho"],
    #             rho_list[0],
    #             rho_list[-1],
    #             (rho_list[1:] - rho_list[:-1]).min().round(2),
    #             best_row["cost"],
    #             best_row["std"],
    #         ]
    #         + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
    #     )


## 2.2. B1-B4

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['B1', 'B2', 'B3', 'B4']:

    if bmk == "B1" or bmk == "B2":
        show_names = (["safety", "rho"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                      + ["cost", "std"]
                      + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (["safety", "rho"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        )
    elif bmk == "B3" or bmk == "B4":
        show_names = (
            ["safety", "rho"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
            + ["cost", "std"]
            + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (
            ["safety", "rho"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
        )
    
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    best_row, rho_list = optimal_check(df)

    df_benchmarks.loc[bmk] = (
        [
            best_row["rho"],
            rho_list[0],
            rho_list[-1],
            (rho_list[1:] - rho_list[:-1]).min().round(2),
            best_row["cost"],
            best_row["std"],
        ]
        + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
    )
    # best_row, rho_list = optimal_check(df[df['safety'] == 1.0])

    # if best_row.empty == False:
    #     df_benchmarks_s1.loc[bmk] = (
    #         [
    #             best_row["rho"],
    #             rho_list[0],
    #             rho_list[-1],
    #             (rho_list[1:] - rho_list[:-1]).min().round(2),
    #             best_row["cost"],
    #             best_row["std"],
    #         ]
    #         + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
    #     )


## 2.3. C1-C2:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ["C1", "C2"]:
    for scheduling_suffix in ["abs", "rel"]: 

        if bmk == "C1":
            show_names = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            )
        elif bmk == "C2":
            show_names = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
            )
    
        
        df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv")
        
        df = df.sort_values(param_cols).reset_index(drop=True)
        
        # display(df)
        print(f'\n\nbmk = {bmk}')
        best_row, rho_list = optimal_check(df)
    
        df_benchmarks.loc[f'{bmk}-{scheduling_suffix}'] = [best_row["rho"], rho_list[0], rho_list[-1], (rho_list[1:] - rho_list[:-1]).min().round(2),  
                                  best_row["cost"], best_row["std"]] + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
        # best_row, rho_list = optimal_check(df[df['safety'] == 1.0])
    
        # if best_row.empty == False:
        #     df_benchmarks_s1.loc[bmk] = (
        #         [
        #             best_row["rho"],
        #             rho_list[0],
        #             rho_list[-1],
        #             (rho_list[1:] - rho_list[:-1]).min().round(2),
        #             best_row["cost"],
        #             best_row["std"],
        #         ]
        #         + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
        #     )


## 2.4. D1-D2:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ["D1", "D2"]:
    for scheduling_suffix in ["abs", "rel"]: 

        if bmk == "D1":
            show_names = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            )
        elif bmk == "D2":
            show_names = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
            )
        
        df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv")
        
        df = df.sort_values(param_cols).reset_index(drop=True)
        
        # display(df)
        print(f'\n\nbmk = {bmk}')
        best_row, rho_list = optimal_check(df)
    
        df_benchmarks.loc[f'{bmk}-{scheduling_suffix}'] = [best_row["rho"], rho_list[0], rho_list[-1], (rho_list[1:] - rho_list[:-1]).min().round(2),  
                                  best_row["cost"], best_row["std"]] + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
        # best_row, rho_list = optimal_check(df[df['safety'] == 1.0])
    
        # if best_row.empty == False:
        #     df_benchmarks_s1.loc[bmk] = (
        #         [
        #             best_row["rho"],
        #             rho_list[0],
        #             rho_list[-1],
        #             (rho_list[1:] - rho_list[:-1]).min().round(2),
        #             best_row["cost"],
        #             best_row["std"],
        #         ]
        #         + [best_row[f"delay{i}"] for i in range(1, dim_K + 2)]
        #     )

                              


# results summary

In [ ]:
display(df_benchmarks)

# 3. Best row by group

## Function:

In [ ]:
def best_by_group(
    df,
    group_col,
    cost_col="cost",
    feasibility_threshold=1,
    sort_groups=True
):
    """
    For each group in group_col:
      1) find the row with minimum cost without constraints
      2) find the row with minimum cost among feasible points
         (delay1 <= threshold and delay2 <= threshold)

    Parameters
    ----------
    df : pd.DataFrame
    group_col : str
        Column to group by, e.g. "rho", "epsilon", "lambda", "safety"
    cost_col : str
        Cost column name
    delay1_col, delay2_col : str
        Delay column names
    feasibility_threshold : float
        Feasibility threshold for both delays
    sort_groups : bool
        Whether to sort output by group_col

    Returns
    -------
    best_unconstrained : pd.DataFrame
    best_constrained : pd.DataFrame
    missing_groups : list
        Groups with no feasible solution
    """

    import pandas as pd

    # required_cols = [group_col, cost_col, delay1_col, delay2_col]
    # missing_cols = [c for c in required_cols if c not in df.columns]
    # if missing_cols:
    #     raise ValueError(f"Missing columns in df: {missing_cols}")

    # group values
    group_values = df[group_col].dropna().unique().tolist()
    if sort_groups:
        try:
            group_values = sorted(group_values)
        except Exception:
            pass

    # # 1) unconstrained best
    idx_unconstrained = df.groupby(group_col)[cost_col].idxmin()
    best_unconstrained = df.loc[idx_unconstrained].copy()

    if sort_groups:
        best_unconstrained = best_unconstrained.sort_values(group_col)

    best_unconstrained = best_unconstrained.reset_index(drop=True)

    # 2) constrained best
    delay_cols = [f"delay{i}" for i in range(1, 12)]
    df_valid = df[(df[delay_cols] <= 1).all(axis=1)].copy()
    # df_valid = df[
    #     (df[delay1_col] <= feasibility_threshold) &
    #     (df[delay2_col] <= feasibility_threshold) &
    #     (df[delay3_col] <= feasibility_threshold) &
    #     (df[delay4_col] <= feasibility_threshold)
    # ].copy()

    best_constrained_list = []
    missing_groups = []

    for g in group_values:
        df_g_valid = df_valid[df_valid[group_col] == g]

        if len(df_g_valid) == 0:
            missing_groups.append(g)
        else:
            best_constrained_list.append(
                df_g_valid.loc[df_g_valid[cost_col].idxmin()]
            )

    if len(best_constrained_list) > 0:
        best_constrained = pd.DataFrame(best_constrained_list)
        if sort_groups:
            best_constrained = best_constrained.sort_values(group_col)
        best_constrained = best_constrained.reset_index(drop=True)
    else:
        best_constrained = pd.DataFrame(columns=df.columns)

    # print(f"Best row by {group_col} (without delay constraints):")
    # display(best_unconstrained)
    
    print(f"Best row by {group_col} (with delay constraints):")
    display(best_constrained)
    display(best_constrained['cost'])
    
    print(f"{group_col} with no feasible solution:")
    print(missing_groups)

    return best_unconstrained, best_constrained, missing_groups

In [ ]:
import matplotlib.pyplot as plt

def plot_best_feasible_cost(best_constrained, group_col, cost_col="cost"):
    if best_constrained.empty:
        print(f"No feasible points for {group_col}")
        return
    
    best_constrained = best_constrained.sort_values(group_col)

    plt.figure(figsize=(6, 4))
    plt.plot(best_constrained[group_col].to_numpy(), best_constrained[cost_col].to_numpy(), marker="o")
    plt.xlabel(group_col)
    plt.ylabel(cost_col)
    plt.title(f"Best feasible {cost_col} vs {group_col}")
    plt.grid(True)
    plt.show()

In [ ]:
def show_best_row_by_group(df, group_col_list):
    for group_col in group_col_list: 
        print("\n")
        print(f"Group by {group_col}:")
        print("\n")
        best_unconstrained, best_constrained, missing = best_by_group(
            df,
            group_col=group_col
        )
        plot_best_feasible_cost(best_constrained, group_col)

## 3.1. A1-A4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ['A1', 'A2', 'A3', 'A4']:

    if bmk == "A1" or bmk == "A2":
        show_names = (["safety", "rho", "epsilon"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                      + ["cost", "std"]
                      + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (["safety", "rho", "epsilon"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        )
    elif bmk == "A3" or bmk == "A4":
        show_names = (
            ["safety", "rho", "epsilon"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
            + ["cost", "std"]
            + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (
            ["safety", "rho", "epsilon"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
        )    
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    group_col_list = ["rho", "epsilon", "safety"]
    show_best_row_by_group(df, group_col_list)


## 3.2. B1-B4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['B1', 'B2', 'B3', 'B4']:

    if bmk == "B1" or bmk == "B2":
        show_names = (["safety", "rho"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                      + ["cost", "std"]
                      + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (["safety", "rho"] 
                      + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
        )
    elif bmk == "B3" or bmk == "B4":
        show_names = (
            ["safety", "rho"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
            + ["cost", "std"]
            + [f"delay{i}" for i in range(1, dim_K + 2)]
        )
        param_cols = (
            ["safety", "rho"]
            + [
                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                for j in range(1, 3 * (dim_K + 1) + 1)
            ]
        )
    
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    group_col_list = ["rho", "safety"]
    show_best_row_by_group(df, group_col_list)


## 3.3. C1-C2:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ["C1", "C2"]:
    for scheduling_suffix in ["abs", "rel"]: 

        if bmk == "C1":
            show_names = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            )
        elif bmk == "C2":
            show_names = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho", "epsilon"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
            )
    
        
        df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv")
        
        df = df.sort_values(param_cols).reset_index(drop=True)
        
        # display(df)
        print(f"\nbmk = {bmk}-{scheduling_suffix}\n")
        
        group_col_list = ["rho", "epsilon", "safety", "lambda", "beta_rate_1", "beta_rate_2"]
        show_best_row_by_group(df, group_col_list)

## 3.4. D1-D2:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

for bmk in ["D1", "D2"]:
    for scheduling_suffix in ["abs", "rel"]: 

        if bmk == "D1":
            show_names = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [f"param_{i}" for i in range(1, 2 * (dim_K + 1) + 1)]
            )
        elif bmk == "D2":
            show_names = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
                          + ["cost", "std"]
                          + [f"delay{i}" for i in range(1, dim_K + 2)]
            )
            param_cols = (["safety", "rho"] 
                          + ["lambda", "beta_rate_1", "beta_rate_2"]
                          + [
                                f"HL_rate_{(j - 1) // 3 + 1}" if (j - 1) % 3 == 0 else f"param_{j}"
                                for j in range(1, 3 * (dim_K + 1) + 1)
                            ]
            )
        
        df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}-{scheduling_suffix}_full.csv")
        
        df = df.sort_values(param_cols).reset_index(drop=True)
        
        # display(df)
        print(f"\nbmk = {bmk}-{scheduling_suffix}\n")
        
        group_col_list = ["rho", "safety", "lambda", "beta_rate_1", "beta_rate_2"]
        show_best_row_by_group(df, group_col_list)